In [ ]:
import sqlite3
import os

DB_PATH = "database.db"

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print(f"Existing {DB_PATH} removed.")

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
cur = conn.cursor()

### OLD
"""
CREATE TABLE IF NOT EXISTS sales_items (
    sale_item_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    sale_id              INTEGER NOT NULL,
    product_id           INTEGER NOT NULL,
    line_number          INTEGER NOT NULL DEFAULT 1,
    base_unit_price      REAL    NOT NULL DEFAULT 0.00,
    sale_unit_price      REAL    NOT NULL DEFAULT 0.00,
    quantity             REAL    NOT NULL DEFAULT 0.000,
    line_base_amount     REAL    NOT NULL DEFAULT 0.00,
    line_sale_amount     REAL    NOT NULL DEFAULT 0.00,
    line_discount_amount REAL    NOT NULL DEFAULT 0.00,
    tax_rate_pct         REAL    NOT NULL DEFAULT 0.00,
    tax_amount           REAL    NOT NULL DEFAULT 0.00,
    promotion_code       TEXT,
    created_at           TEXT    NOT NULL DEFAULT (strftime('%Y-%m-%dT%H:%M:%SZ', 'now')),
    FOREIGN KEY (sale_id)    REFERENCES sales    (sale_id),
    FOREIGN KEY (product_id) REFERENCES products (product_id)
);

"""


schema_sql = """
CREATE TABLE IF NOT EXISTS suppliers (
    supplier_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    supplier_code   TEXT    NOT NULL UNIQUE,
    supplier_name   TEXT    NOT NULL,
    contact_name    TEXT,
    contact_email   TEXT,
    contact_phone   TEXT,
    address         TEXT,
    is_active       INTEGER NOT NULL DEFAULT 1 CHECK (is_active IN (0, 1)),
    created_at      TEXT    NOT NULL DEFAULT (strftime('%Y-%m-%dT%H:%M:%SZ', 'now'))
);

CREATE TABLE IF NOT EXISTS products (
    product_id          INTEGER PRIMARY KEY AUTOINCREMENT,
    product_sku         TEXT    NOT NULL UNIQUE,
    product_name        TEXT    NOT NULL,
    brand               TEXT,
    category            TEXT,
    supplier_id         INTEGER NOT NULL,
    unit_of_measure     TEXT,
    pack_size           TEXT,
    base_cost           REAL    NOT NULL DEFAULT 0.00,
    base_retail_price   REAL    NOT NULL DEFAULT 0.00,
    tax_rate_pct        REAL    NOT NULL DEFAULT 0.00,
    is_discontinued     INTEGER NOT NULL DEFAULT 0 CHECK (is_discontinued IN (0, 1)),
    created_at          TEXT    NOT NULL DEFAULT (strftime('%Y-%m-%dT%H:%M:%SZ', 'now')),
    FOREIGN KEY (supplier_id) REFERENCES suppliers (supplier_id)
);

CREATE TABLE IF NOT EXISTS orders (
    order_id        INTEGER PRIMARY KEY AUTOINCREMENT,
    order_number    TEXT    NOT NULL,
    order_date      TEXT    NOT NULL,
    supplier_id     INTEGER NOT NULL,
    product_id      INTEGER NOT NULL,
    ordered_qty     REAL    NOT NULL DEFAULT 0.000,
    received_qty    REAL             DEFAULT 0.000,
    unit_cost       REAL    NOT NULL DEFAULT 0.00,
    line_base_cost  REAL             DEFAULT 0.00,
    discount_pct    REAL             DEFAULT 0.00,
    tax_rate_pct    REAL             DEFAULT 0.00,
    status          TEXT    NOT NULL DEFAULT 'CREATED'
                            CHECK (status IN ('CREATED', 'RECEIVED', 'CANCELLED')),
    created_at      TEXT    NOT NULL DEFAULT (strftime('%Y-%m-%dT%H:%M:%SZ', 'now')),
    FOREIGN KEY (supplier_id) REFERENCES suppliers (supplier_id),
    FOREIGN KEY (product_id)  REFERENCES products  (product_id)
);


CREATE TABLE IF NOT EXISTS sales_items (
    sale_item_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    sale_id              INTEGER NOT NULL,              -- kept as a plain column, no FK
    product_id           INTEGER NOT NULL,
    line_number          INTEGER NOT NULL DEFAULT 1,
    base_unit_price      REAL    NOT NULL DEFAULT 0.00,
    sale_unit_price      REAL    NOT NULL DEFAULT 0.00,
    quantity             REAL    NOT NULL DEFAULT 0.000,
    line_base_amount     REAL    NOT NULL DEFAULT 0.00,
    line_sale_amount     REAL    NOT NULL DEFAULT 0.00,
    line_discount_amount REAL    NOT NULL DEFAULT 0.00,
    tax_rate_pct         REAL    NOT NULL DEFAULT 0.00,
    tax_amount           REAL    NOT NULL DEFAULT 0.00,
    promotion_code       TEXT,
    created_at           TEXT    NOT NULL DEFAULT (strftime('%Y-%m-%dT%H:%M:%SZ', 'now')),
    FOREIGN KEY (product_id) REFERENCES products (product_id)   -- this one stays
);

CREATE TABLE IF NOT EXISTS inventory (
    inventory_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id       INTEGER NOT NULL UNIQUE,
    quantity_on_hand REAL    NOT NULL DEFAULT 0.000,
    reorder_point    REAL             DEFAULT 0.000,
    reorder_qty      REAL             DEFAULT 0.000,
    last_updated_at  TEXT    NOT NULL DEFAULT (strftime('%Y-%m-%dT%H:%M:%SZ', 'now')),
    FOREIGN KEY (product_id) REFERENCES products (product_id)
);
"""

cur.executescript(schema_sql)
conn.commit()
conn.close()

print(f"✅ {DB_PATH} created successfully.")

✅ database.db created successfully.


In [6]:
import sqlite3
import pandas as pd

DB_PATH = "database.db"

conn = sqlite3.connect(DB_PATH)

# Get all table names
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", conn)
print("=== Tables in database ===")
print(tables.to_string(index=False))
print()

# For each table, show its schema and an empty preview
for table in tables["name"]:
    print(f"{'='*55}")
    print(f"  TABLE: {table}")
    print(f"{'='*55}")
    
    # Column info via PRAGMA
    pragma = pd.read_sql(f"PRAGMA table_info({table});", conn)
    pragma = pragma[["cid", "name", "type", "notnull", "dflt_value", "pk"]]
    pragma.columns = ["#", "Column", "Type", "NotNull", "Default", "PK"]
    print(pragma.to_string(index=False))
    
    # Row count + empty dataframe preview
    count = pd.read_sql(f"SELECT COUNT(*) AS row_count FROM {table};", conn).iloc[0, 0]
    df_preview = pd.read_sql(f"SELECT * FROM {table} LIMIT 5;", conn)
    print(f"\n  Rows: {count}")
    print(f"  Columns: {list(df_preview.columns)}")
    print()

conn.close()

=== Tables in database ===
           name
      inventory
         orders
       products
    sales_items
sqlite_sequence
      suppliers

  TABLE: inventory
 #           Column    Type  NotNull                               Default  PK
 0     inventory_id INTEGER        0                                   NaN   1
 1       product_id INTEGER        1                                   NaN   0
 2 quantity_on_hand    REAL        1                                 0.000   0
 3    reorder_point    REAL        0                                 0.000   0
 4      reorder_qty    REAL        0                                 0.000   0
 5  last_updated_at    TEXT        1 strftime('%Y-%m-%dT%H:%M:%SZ', 'now')   0

  Rows: 0
  Columns: ['inventory_id', 'product_id', 'quantity_on_hand', 'reorder_point', 'reorder_qty', 'last_updated_at']

  TABLE: orders
 #         Column    Type  NotNull                               Default  PK
 0       order_id INTEGER        0                                   N

In [7]:
import sqlite3
import pandas as pd
import os

DB_PATH = "database.db"
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

# ── Helpers ────────────────────────────────────────────────────────────────────

def normalize_date(val):
    """Try to parse various date formats and return ISO 8601 TEXT."""
    if pd.isna(val) or val == "":
        return None
    for fmt in ("%d/%m/%Y %H:%M", "%m/%d/%Y %H:%M", "%m/%d/%Y",
                "%Y-%m-%d %H:%M:%S", "%Y-%m-%d"):
        try:
            return pd.to_datetime(val, format=fmt).strftime("%Y-%m-%dT%H:%M:%SZ")
        except (ValueError, TypeError):
            continue
    # fallback: let pandas infer
    try:
        return pd.to_datetime(val, dayfirst=False).strftime("%Y-%m-%dT%H:%M:%SZ")
    except Exception:
        return str(val)

def bool_to_int(val):
    """Convert True/False strings or booleans to 1/0."""
    if isinstance(val, bool):
        return int(val)
    if isinstance(val, str):
        return 1 if val.strip().lower() == "true" else 0
    return int(val) if pd.notna(val) else 0

def clean_email(val):
    """Strip markdown link syntax from email, e.g. [email](mailto:email) → email."""
    if pd.isna(val):
        return None
    import re
    match = re.match(r'\[([^\]]+)\]\(mailto:[^\)]+\)', str(val))
    return match.group(1) if match else str(val)

# ── 1. suppliers ───────────────────────────────────────────────────────────────

df_sup = pd.read_csv("suppliers.csv")
df_sup["is_active"]   = df_sup["is_active"].apply(bool_to_int)
df_sup["contact_email"] = df_sup["contact_email"].apply(clean_email)
df_sup["created_at"]  = df_sup["created_at"].apply(normalize_date)

df_sup.to_sql("suppliers", conn, if_exists="append", index=False)
print(f"✅ suppliers: {len(df_sup)} row(s) inserted")

# ── 2. products ────────────────────────────────────────────────────────────────

df_prod = pd.read_csv("products.csv")
df_prod["is_discontinued"] = df_prod["is_discontinued"].apply(bool_to_int)
df_prod["created_at"]      = df_prod["created_at"].apply(normalize_date)

df_prod.to_sql("products", conn, if_exists="append", index=False)
print(f"✅ products: {len(df_prod)} row(s) inserted")

# ── 3. inventory ───────────────────────────────────────────────────────────────

df_inv = pd.read_csv("inventory.csv")
# Drop trailing blank columns (unnamed)
df_inv = df_inv.loc[:, ~df_inv.columns.str.startswith("Unnamed")]
df_inv["last_updated_at"] = df_inv["last_updated_at"].apply(normalize_date)

df_inv.to_sql("inventory", conn, if_exists="append", index=False)
print(f"✅ inventory: {len(df_inv)} row(s) inserted")

# ── 4. orders ──────────────────────────────────────────────────────────────────

df_ord = pd.read_csv("orders.csv")
df_ord["order_date"]  = df_ord["order_date"].apply(normalize_date)
df_ord["created_at"]  = df_ord["created_at"].apply(normalize_date)

# Drop columns not present in the schema
df_ord.drop(columns=["expected_delivery_date", "actual_delivery_date"],
            errors="ignore", inplace=True)

# Remap any status values not allowed by the CHECK constraint
status_map = {
    "SENT":      "CREATED",   # closest equivalent — adjust if needed
    "PENDING":   "CREATED",
    "DELIVERED": "RECEIVED",
    "CLOSED":    "RECEIVED",
}
df_ord["status"] = df_ord["status"].replace(status_map)

df_ord.to_sql("orders", conn, if_exists="append", index=False)
print(f"✅ orders: {len(df_ord)} row(s) inserted")

# ── 5. sales_items ─────────────────────────────────────────────────────────────

df_sales = pd.read_csv("sales_transaction_lines.csv")
df_sales["created_at"] = df_sales["created_at"].apply(normalize_date)

df_sales.to_sql("sales_items", conn, if_exists="append", index=False)
print(f"✅ sales_items: {len(df_sales)} row(s) inserted")

# ── Commit & close ─────────────────────────────────────────────────────────────

conn.commit()
conn.close()
print("\n🎉 All CSVs loaded into database.db successfully.")

✅ suppliers: 10 row(s) inserted
✅ products: 150 row(s) inserted
✅ inventory: 150 row(s) inserted


/tmp/ipykernel_108596/1177186724.py:23: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(val, dayfirst=False).strftime("%Y-%m-%dT%H:%M:%SZ")


✅ orders: 1500 row(s) inserted


DatabaseError: Execution failed